# How PyPSA builds its model from a spec (PR #2128)

> **Throwaway notebook for reviewers.** It is not part of the release or the docs.

This PR changes how `n.optimize()` builds its linopy model.
Before, Python code wrote every variable and constraint.
Now, YAML files state the math, and linopy builds the model from them.

The notebook has two parts:

1. **How it works.** The principles, shown on a small network.
2. **How it is built.** The Python modules and what each one does.

You need no knowledge of `mathspec`. Every cell runs on the PR branch.

In [1]:
import logging
import warnings
from importlib.resources import files

import linopy
import pandas as pd
import yaml

import pypsa

pypsa.options.future.string_dtype = False
pypsa.options.future.exclude_objective_constant = True
warnings.filterwarnings("ignore", category=linopy.EvolvingAPIWarning)
for logger in ("pypsa", "linopy"):
    logging.getLogger(logger).setLevel(logging.ERROR)

SPEC_DIR = files("pypsa.optimization.spec")

# Part 1: How it works

## 1.1 The idea in one picture

```
  YAML files (the math)          pypsa.Network (the data)
          │                               │
          ▼                               ▼
   one Spec object            NetworkSources (a lazy dict)
          │                               │
          └───────────► linopy ◄──────────┘
                   Model.from_spec(spec, sources)
                            │
                            ▼
                      linopy.Model
```

Three rules hold the design together:

| Rule | What it means |
|---|---|
| The math lives only in YAML | Every variable, constraint and objective term is written in a `.yaml` file. No Python function adds a constraint. |
| Python assembles and prepares | Python joins the YAML files into one spec and turns network data into the arrays the spec asks for. |
| linopy builds | linopy reads the spec and the arrays and creates the model. PyPSA only renames and scales the result. |

## 1.2 A small network

We use a two-bus network: wind in the north, gas and a battery in the south.
Wind and gas are extendable, so the model chooses their capacity.

In [2]:
def toy() -> pypsa.Network:
    n = pypsa.Network()
    n.set_snapshots(pd.date_range("2030-01-01", periods=4, freq="h"))
    n.add("Carrier", ["wind", "gas", "AC"], co2_emissions=[0, 0.4, 0])
    n.add("Bus", ["north", "south"], carrier="AC")
    n.add("Line", "north-south", bus0="north", bus1="south", x=0.1, r=0.01, s_nom=50, carrier="AC")
    n.add("Load", "demand", bus="south", p_set=[60, 80, 100, 70])
    n.add(
        "Generator", "wind", bus="north", carrier="wind",
        p_nom_extendable=True, capital_cost=80, p_max_pu=[0.9, 0.5, 0.2, 0.7],
    )
    n.add(
        "Generator", "gas", bus="south", carrier="gas",
        p_nom_extendable=True, capital_cost=40, marginal_cost=60,
    )
    n.add("Store", "battery", bus="south", e_nom_extendable=True, capital_cost=10, e_cyclic=True)
    return n


n = toy()
n.optimize(log_to_console=False)
print(f"objective: {n.objective:,.0f}")
n.generators.p_nom_opt.round(1)

objective: 17,414


name
wind    71.4
gas     40.0
Name: p_nom_opt, dtype: float64

For the user, nothing has changed. `n.optimize()` returns the same result as before.
The rest of this notebook looks at what happens inside that call.

## 1.3 The spec is data

`pypsa.optimization.spec.spec()` returns the whole optimisation problem as one object.
It is plain data: you can inspect it, print it and compare it.

In [3]:
from pypsa.optimization.spec import spec

s = spec()
sections = ("dimensions", "relations", "parameters", "variables", "expressions", "constraints")
pd.Series({k: len(getattr(s, k)) for k in sections}, name="entries")

dimensions      20
relations       28
parameters     326
variables       70
expressions    158
constraints    235
Name: entries, dtype: int64

In [4]:
print("objective:", s.objective.sense, s.objective.expression)
print()
print(yaml.safe_dump({"total_cost": s.to_dict()["expressions"]["total_cost"]}, sort_keys=False))

objective: minimize total_cost

total_cost:
  dims: []
  expression: Generator_capex + Link_capex + StorageUnit_capex + Store_capex + Line_capex
    + Process_capex + Transformer_capex + risk_weighted_opex
  description: capital cost of every component, plus the risk-weighted operating cost



The objective reads like the formula in a paper. Each `*_capex` term is itself an entry of the spec.

## 1.4 Write once, use for every component

Many components share the same math. For example, Generator, Link and Store all have
an investment cost. The spec writes such math once, in a **trait** file under `traits/`.

A trait uses **placeholders** in upper case:

| Placeholder | Means | Generator | Store |
|---|---|---|---|
| `COMPONENT` | component name | `Generator` | `Store` |
| `UNIT` | unit dimension | `Generator` | `Store` |
| `NOM` | nominal capacity | `p_nom` | `e_nom` |
| `POWER` | power variable | `p` | `p` |
| `LEVEL` | storage level | (none) | `e` |

Here is the investment cost as it is written in `traits/capacity.yaml`:

In [5]:
def raw(path: str, section: str, key: str) -> None:
    entry = yaml.safe_load((SPEC_DIR / path).read_text())[section][key]
    print(yaml.safe_dump({key: entry}, sort_keys=False, width=100))


raw("traits/capacity.yaml", "parameters", "COMPONENT_capital_weight")
raw("traits/capacity.yaml", "expressions", "COMPONENT_capex")

COMPONENT_capital_weight:
  description: the summed weights of the periods the asset is active in
  dims:
  - UNIT

COMPONENT_capex:
  expression: sum(scenario_weight * COMPONENT_build_cost * COMPONENT_capital_weight)



And here is what the rendered spec holds for Generator and for Store:

In [6]:
def rendered(section: str, *keys: str) -> None:
    entries = s.to_dict()[section]
    print(yaml.safe_dump({k: entries[k] for k in keys}, sort_keys=False, width=100))


rendered("expressions", "Generator_capex", "Store_capex")
rendered("parameters", "Generator_p_nom_max", "Store_e_nom_max")

Generator_capex: sum(scenario_weight * Generator_build_cost * Generator_capital_weight)
Store_capex: sum(scenario_weight * Store_build_cost * Store_capital_weight)

Generator_p_nom_max:
  dims:
  - Generator
  description: most nominal capacity of an extendable asset, the smallest over the scenarios, as every
    scenario shares the build
Store_e_nom_max:
  dims:
  - Store
  description: most nominal capacity of an extendable asset, the smallest over the scenarios, as every
    scenario shares the build



`NOM` became `p_nom` for the Generator and `e_nom` for the Store.
One trait file gives one copy of the math per component.

Each component lists its traits in its class, as `_formulation`:

In [7]:
from pypsa.optimization.spec._render import formulated, symbols

pd.DataFrame(
    {c.name: {t: "x" for t in c._formulation} for c in formulated(pypsa.Network())}
).fillna("").T

,capacity,marginal_cost,dispatch,oneport,rating,commitment,ramping,maintenance,operational_limit,transmission_expansion,multiport,storage,passive_branch
Generator,x,x,x,x,x,x,x,x,x,,,,
Load,,,,x,,,,,,,,,
Link,x,x,x,,x,x,x,x,,x,x,,
Store,x,x,x,x,,,,,x,,,x,
StorageUnit,x,x,,x,,,,,x,,,x,
Line,x,,,,,,,,,x,,,x
Process,x,x,x,,x,x,x,x,,,x,,
Transformer,x,,,,,,,,,,,,x


## 1.5 Spec names and PyPSA names

A spec key uses `_` only, such as `Generator_ext_p_upper`.
PyPSA has always named its constraints with `-`, such as `Generator-ext-p-upper`.
A YAML entry keeps the old name in its `name:` field:

In [8]:
raw("traits/rating.yaml", "constraints", "COMPONENT_ext_p_upper")

COMPONENT_ext_p_upper:
  name: COMPONENT-ext-p-upper
  description: an extendable unit that is not committable runs at most at its maximum of the chosen build
  dims:
  - scenario
  - snapshot
  - UNIT
  where: COMPONENT_ext AND NOT COMPONENT_committable
  expression: COMPONENT_p <= COMPONENT_p_max_pu * (COMPONENT_NOM_ext - COMPONENT_maintenance_pu * COMPONENT_maintenance_capacity)



In [9]:
from pypsa.optimization.spec.build import spec_names

names = spec_names()["constraints"]
print("Generator_ext_p_upper ->", names["Generator_ext_p_upper"])
print("model has it:", "Generator-ext-p-upper" in n.model.constraints)

Generator_ext_p_upper -> Generator-ext-p-upper
model has it: True


So existing code that reads `n.model.constraints["Generator-ext-p-upper"]` keeps working.

## 1.6 Where the numbers come from: `NetworkSources`

The spec says *what* a parameter is. It does not say where its values come from.
`NetworkSources` answers that. It behaves like a dict:

- Its **keys** are the names the spec declares.
- Its **values** are computed only when linopy asks for them, and then cached.

We build one by hand here. `build_model` does the same inside `n.optimize()`.

In [10]:
from pypsa.optimization.spec import NetworkSources
from pypsa.optimization.window import SnapshotWindow


def sources(n: pypsa.Network, multi_invest: bool = False) -> NetworkSources:
    n.calculate_dependent_values()
    window = SnapshotWindow.build(n, n.snapshots, pypsa.options.optimization.model_snapshot_index)
    return NetworkSources(n, spec(), window.network_index, window.model_index, multi_invest=multi_invest)


src = sources(toy())
print("declared keys:", len(src))
print("computed so far:", len(src._cache))

declared keys: 374
computed so far: 0


In [11]:
src["Generator_capital_weight"].to_pandas()

Generator
wind    1.0
gas     1.0
Name: Generator_capital_weight, dtype: float64

In [12]:
print("computed so far:", list(src._cache))

computed so far: ['Generator', 'Generator labels', 'Generator active', 'Generator active by period', 'Generator_capital_weight']


One request computed a few keys, not all of them.
The derivation asked for the activity mask, and the cache kept it for the next caller.

A key read straight from a PyPSA attribute needs no Python at all:

In [13]:
cost = src["Generator_marginal_cost"]
print(cost.dims)
cost.isel(scenario=0, drop=True).to_pandas()

('Generator', 'scenario', 'snapshot')


snapshot,2030-01-01 00:00:00,2030-01-01 01:00:00,2030-01-01 02:00:00,2030-01-01 03:00:00
Generator,,,,
wind,0.0,0.0,0.0,0.0
gas,60.0,60.0,60.0,60.0


A key that nobody can supply fails loudly:

In [14]:
try:
    src["Generator_foo"]
except NotImplementedError as error:
    print(f"NotImplementedError: {error}")

NotImplementedError: no data prep for Generator_foo


## 1.7 Change the problem without writing Python: `overrides`

`n.optimize(overrides=[...])` lays a patch over the spec.
A patch can remove an entry (`null`), replace it, or add a new one.
It can be a dict, YAML text or a path to a YAML file.

**Example 1: replace a constraint.** Extendable generators may run at only half their capacity.

In [15]:
HALF = {
    "constraints": {
        "Generator_ext_p_upper": None,
        "Generator_p_half": {
            "dims": ["Generator", "scenario", "snapshot"],
            "where": "Generator_p_nom_extendable AND not Generator_committable AND Generator_active",
            "expression": "Generator_p <= 0.5 * Generator_p_max_pu * Generator_p_nom_ext",
        },
    }
}

n = toy()
n.optimize(log_to_console=False)
base = n.objective
n.optimize(overrides=[HALF], log_to_console=False)
print(f"objective: {base:,.0f} -> {n.objective:,.0f}")
print([c for c in n.model.constraints if c.startswith("Generator")])

objective: 17,414 -> 23,967
['Generator-ext-p-lower', 'Generator-ext-p_nom-lower', 'Generator_p_half']


**Example 2: add a parameter and a constraint.** Wind must run at least a given number of full-load hours.
The new parameter `Generator_full_load_hours_min` follows the `<Component>_<attr>` pattern.
So `NetworkSources` reads it from a new column of `n.c.generators.static`. No Python is needed.

In [16]:
FULL_LOAD_HOURS = '''
parameters:
  Generator_full_load_hours_min:
    dims: [Generator]
constraints:
  Generator_full_load_hours:
    dims: [scenario, Generator]
    where: Generator_full_load_hours_min > 0 AND Generator_p_nom_extendable
    expression: >-
      sum(Generator_p * snapshot_weighting_generators, over=snapshot)
      >= Generator_full_load_hours_min * Generator_p_nom_ext
'''

rows = {}
for hours in (0.0, 2.0, 2.2):
    n = toy()
    n.c.generators.static["full_load_hours_min"] = [hours, 0.0]
    n.optimize(overrides=[FULL_LOAD_HOURS], log_to_console=False)
    rows[hours] = {"objective": n.objective, **n.generators.p_nom_opt.round(1).to_dict()}
pd.DataFrame(rows).T.rename_axis("wind full-load hours")

,objective,wind,gas
wind full-load hours,,,
0.0,17414.285714,71.4,40.0
2.0,17414.285714,71.4,40.0
2.2,17575.000000,62.5,43.1


Wind can deliver at most 2.3 full-load hours in this network.
At 2.0 the new constraint does not bind. At 2.2 it does: the model builds less wind and more gas.

## 1.8 Linearised unit commitment is a patch, too

`linearized_unit_commitment=True` is not a code branch. It is a folder of YAML files,
`linearized_uc/`, laid over the base spec. Compare the status variable:

In [17]:
pd.DataFrame(
    {
        label: {
            "domain": status.domain,
            "upper bound": status.bounds.upper,
            "description": status.description,
        }
        for label, status in (
            ("default", spec().variables["Generator_status"]),
            ("linearized_unit_commitment", spec(True).variables["Generator_status"]),
        )
    }
)

,default,linearized_unit_commitment
domain,integer,continuous
upper bound,NaN,1.0
description,how many modules of a committable unit are on;...,the share of a committable unit that is on


## 1.9 Trace one key from start to end: `capital_weight`

`Generator_capital_weight` answers one question:
**how often does a unit's capital cost count in the objective?**

| Step | Where | What happens |
|---|---|---|
| 1. Declare | `traits/capacity.yaml` | `COMPONENT_capital_weight` is declared with placeholders and used in `COMPONENT_capex`. |
| 2. Render | `_render.py` | One copy per component: `Generator_capital_weight`, `Store_capital_weight`, ... |
| 3. Derive | `sources.py` | `@source("COMPONENT_capital_weight")` computes the values. |
| 4. Build | `build.py` | linopy evaluates `Generator_capex` with these values. |

The derivation in step 3:

In [18]:
import inspect

from pypsa.optimization.spec import sources as sources_module

print(inspect.getsource(sources_module._capital_weight))

@source("COMPONENT_capital_weight")
def _capital_weight(s: NetworkSources, name: str, c: Components) -> xr.DataArray:
    by_period = s.active_by_period(c)
    if not s.multi_invest:
        return by_period.any("period").astype(float)
    return (by_period * s["period_weighting_objective"]).sum("period")



With one period, the weight is 1 for an active unit. It gets interesting with investment periods.
Here, three periods weigh 1.0, 0.7 and 0.5 in the objective. Three generators live through different periods:

In [19]:
def periods() -> pypsa.Network:
    n = pypsa.Network()
    n.set_snapshots(pd.MultiIndex.from_product([[2030, 2040, 2050], range(2)]))
    n.set_investment_periods([2030, 2040, 2050])
    n.investment_period_weightings["objective"] = [1.0, 0.7, 0.5]
    n.add("Carrier", ["wind", "gas"])
    n.add("Bus", "home")
    n.add("Load", "demand", bus="home", p_set=50)
    n.add("Generator", "wind-2030", bus="home", carrier="wind", build_year=2030, lifetime=15,
          p_nom_extendable=True, capital_cost=80)
    n.add("Generator", "wind-2040", bus="home", carrier="wind", build_year=2040, lifetime=30,
          p_nom_extendable=True, capital_cost=60)
    n.add("Generator", "gas", bus="home", carrier="gas", build_year=2030, lifetime=50,
          p_nom_extendable=True, capital_cost=40, marginal_cost=60)
    return n


n = periods()
src = sources(n, multi_invest=True)
active = src.active_by_period(n.components.generators).to_pandas().T
active["capital_weight"] = src["Generator_capital_weight"].to_pandas()
active

period,2030,2040,2050,capital_weight
Generator,,,,
wind-2030,True,True,False,1.7
wind-2040,False,True,True,1.2
gas,True,True,True,2.2


`wind-2030` is active in 2030 and 2040, so its weight is 1.0 + 0.7 = 1.7.

Now we check that the built model uses exactly this weight.
The objective coefficient of `Generator-p_nom` must equal `capital_cost × capital_weight`:

In [20]:
n.optimize(multi_investment_periods=True, log_to_console=False)
m = n.model
coefficients = m.objective.flat.groupby("vars").coeffs.sum()
labels = m.variables["Generator-p_nom"].labels.to_pandas()
pd.DataFrame(
    {
        "capital_cost × capital_weight": n.generators.capital_cost * active["capital_weight"],
        "objective coefficient in model": labels.map(coefficients),
    }
)

,capital_cost × capital_weight,objective coefficient in model
name,,
wind-2030,136.0,136.0
wind-2040,72.0,72.0
gas,88.0,88.0


The two columns match. The YAML formula, the Python derivation and the built model agree.

**A question for you:** suppose the capital cost should count only in the period a unit is *built*.
Would you change the YAML or the `@source` function?
(Hint: `Generator_first_active` exists already.)

## 1.10 Render the math as LaTeX

The spec is data, so `mathspec` can print it, too. Two functions do this:

| Function | Prints |
|---|---|
| `mathspec.typeset_line(spec, key, fmt)` | One entry: a constraint, an expression or a variable's domain |
| `mathspec.typeset(spec, fmt)` | The whole spec as one document, with sets, symbols and every equation |

`fmt` is `"latex"`, `"markdown"` or `"typst"`.

A symbol table decides how each name prints. PyPSA keeps it in `symbols.yaml`, next to the spec:

In [21]:
import shutil
import subprocess
import tempfile
from pathlib import Path

import mathspec
from IPython.display import Math, display

notation = yaml.safe_load((SPEC_DIR / "symbols.yaml").read_text())
table = mathspec.SymbolTable.load(notation)
{k: notation["names"][k] for k in ("Generator_p", "Generator_p_nom_ext", "Generator_p_max_pu")}

{'Generator_p': 'g',
 'Generator_p_nom_ext': 'G',
 'Generator_p_max_pu': '\\bar{g}'}

### One entry

`typeset_line` gives the bare LaTeX of one entry. Jupyter renders it with MathJax.
The quantifier after `\forall` lists the sets the row runs over, and the condition after `:` is its `where:`.

In [22]:
for key in ("Generator_capex", "Generator_ext_p_upper"):
    display(Math(mathspec.typeset_line(spec(), key, "latex", symbols=table)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Without the table, `mathspec` makes a symbol from each name. It is readable, but long.
The same line in Typst shows that the format is only a parameter:

In [23]:
display(Math(mathspec.typeset_line(spec(), "Generator_capex", "latex")))
print(mathspec.typeset_line(spec(), "Generator_capex", "typst"))

<IPython.core.display.Math object>

italic("Generator_capex") = sum_(g in cal(G), j in cal(J)) upright("scenario_weight")_(j) dot italic("Generator_build_cost")_(g,j) dot upright("Generator_capital_weight")_(g)


An override is math like any other entry. Here is the constraint that `HALF` added in 1.7:

In [24]:
display(Math(mathspec.typeset_line(spec(overrides=[HALF]), "Generator_p_half", "latex", symbols=table)))

<IPython.core.display.Math object>

### The whole spec as a PDF

`typeset(..., standalone=True)` gives a LaTeX document you can compile.
The cell writes it to a temporary folder and compiles it with `lualatex` if you have it.
Use `lualatex`, not `pdflatex`: one description holds a `π`, which `pdflatex` rejects.

In [25]:
out = Path(tempfile.mkdtemp())
tex = out / "pypsa-formulation.tex"
tex.write_text(mathspec.typeset(spec(), "latex", symbols=table, standalone=True))
print(tex, f"({len(tex.read_text().splitlines())} lines)")
if shutil.which("lualatex"):
    subprocess.run(["lualatex", "-interaction=nonstopmode", "-halt-on-error", tex.name], cwd=out, check=True, capture_output=True)
    print(tex.with_suffix(".pdf"))

/tmp/tmpjtvbtqpv/pypsa-formulation.tex (1446 lines)


/tmp/tmpjtvbtqpv/pypsa-formulation.pdf


The docs pages of the formulation use the same functions; section 2.9 shows them.

# Part 2: How it is built

## 2.1 The pipeline

```
YAML files ──_render.py──► one Spec ──build.py──► linopy Model
                                          ▲
pypsa.Network ──sources.py + data-prep modules──┘
                 (checks.py runs first)
```

`n.optimize()` calls `build_model` in `build.py`. Every other module feeds that one call.

## 2.2 The modules

| Module | Role |
|---|---|
| `_render.py` | Joins `system.yaml`, every trait a component declares (with placeholders filled) and the component's own file into one `Spec`. Collects the PyPSA names (`name:`) and scaling rules (`scaling:`). |
| `build.py` | `spec()`: base spec plus optional patches (linearised UC, piecewise options, `overrides`), cached. `build_model()`: check, spec, sources, `Model.from_spec`, then rename and scale. |
| `sources.py` | `NetworkSources`, the lazy dict of all data. The `@source` registry. General derivations (activity, weights, ports, ...). |
| `_global_constraints.py` | Data prep for global constraints and carrier growth limits. |
| `_passive_network.py` | Data prep for KVL cycles, loss cuts, outage factors and flow-based domains. |
| `_piecewise.py` | Data prep for piecewise curves (breakpoint tables). |
| `_unit_features.py` | Data prep for maintenance. |
| `checks.py` | Warnings and errors on the network before the build, as PyPSA raised them before. |
| `_docs.py` | Renders the spec as the formulation pages of the docs. |

In [26]:
from pathlib import Path

root = Path(str(SPEC_DIR))
pd.DataFrame(
    {
        "lines": {p.name: len(p.read_text().splitlines()) for p in sorted(root.glob("*.py"))},
    }
).sort_values("lines", ascending=False)

,lines
sources.py,1007
build.py,562
_docs.py,546
_passive_network.py,394
_global_constraints.py,321
_render.py,258
checks.py,256
_piecewise.py,113
_unit_features.py,86
__init__.py,16


In [27]:
yaml_files = sorted(p.relative_to(root).as_posix() for p in root.rglob("*.yaml"))
print(len(yaml_files), "YAML files:")
print("\n".join(yaml_files))

27 YAML files:
components/generator.yaml
components/line.yaml
components/link.yaml
components/load.yaml
components/process.yaml
components/storage_unit.yaml
components/store.yaml
components/transformer.yaml
linearized_uc/components/generator.yaml
linearized_uc/system.yaml
linearized_uc/traits/commitment.yaml
linearized_uc/traits/multiport.yaml
symbols.yaml
system.yaml
traits/capacity.yaml
traits/commitment.yaml
traits/dispatch.yaml
traits/maintenance.yaml
traits/marginal_cost.yaml
traits/multiport.yaml
traits/oneport.yaml
traits/operational_limit.yaml
traits/passive_branch.yaml
traits/ramping.yaml
traits/rating.yaml
traits/storage.yaml
traits/transmission_expansion.yaml


## 2.3 Rendering: which files make the spec

`render()` reads files in a fixed order: `system.yaml` first, then per component
each of its trait files and its own file under `components/`.
A trait file is renamed for each component that declares it.
Files are joined as a dict. Two files that declare the same entry raise a `ValueError`.

This is the reading order for the Store:

In [28]:
from pypsa.optimization.spec._render import _snake

store = pypsa.Network().components.stores
paths = [f"traits/{t}.yaml" for t in store._formulation] + [f"components/{_snake(store.name)}.yaml"]
pd.DataFrame({"file": paths, "exists": [(SPEC_DIR / p).is_file() for p in paths]})

,file,exists
0,traits/capacity.yaml,True
1,traits/marginal_cost.yaml,True
2,traits/dispatch.yaml,True
3,traits/oneport.yaml,True
4,traits/operational_limit.yaml,True
5,traits/storage.yaml,True
6,components/store.yaml,True


In [29]:
pd.DataFrame({c.name: symbols(c) for c in formulated(pypsa.Network())}).fillna("").T

,COMPONENT,UNIT,NOM,POWER,COEF,CYCLIC,LEVEL
Generator,Generator,Generator,p_nom,p,,,
Load,Load,Load,p_nom,,,,
Link,Link,Link,p_nom,p,efficiency,,
Store,Store,Store,e_nom,p,,e_cyclic,e
StorageUnit,StorageUnit,StorageUnit,p_nom,p_dispatch,,cyclic_state_of_charge,state_of_charge
Line,Line,Line,s_nom,,,,
Process,Process,Process,p_nom,p,rate,,
Transformer,Transformer,Transformer,s_nom,,,,


A component lacks a placeholder it has no value for. A Load has no `POWER`, for example.
A trait that uses that placeholder fails to render for it, so a wrong trait list fails early.

## 2.4 The `@source` registry

Data-prep functions register themselves with a decorator:

```python
@source("COMPONENT_capital_weight")
def _capital_weight(s: NetworkSources, name: str, c: Components) -> xr.DataArray: ...
```

The four `_*.py` data-prep modules exist only to register such functions.
`spec/__init__.py` imports them for that reason alone.

In [30]:
from pypsa.optimization.spec.sources import _SOURCES

registry = pd.DataFrame(
    {
        "module": [f.__module__.rsplit(".", 1)[-1] for f in _SOURCES.values()],
        "placeholder key": ["COMPONENT" in k or "UNIT" in k for k in _SOURCES],
    },
    index=list(_SOURCES),
)
print(len(registry), "registered keys")
registry.groupby("module")["placeholder key"].agg(["count", "sum"]).rename(
    columns={"count": "keys", "sum": "with placeholders"}
)

105 registered keys


,keys,with placeholders
module,,
_global_constraints,12,5
_passive_network,22,6
_piecewise,11,4
_unit_features,4,4
sources,56,36


A key with placeholders stands for one key per component. `_providers` expands it:
for each component that has all the placeholders, it fills them in and binds the component as `c`.

In [31]:
from pypsa.optimization.spec.sources import _providers

providers = _providers(formulated(pypsa.Network()))
print(len(_SOURCES), "registered keys ->", len(providers), "concrete keys")
sorted(k for k in providers if k.endswith("_capital_weight"))

105 registered keys -> 478 concrete keys


['Generator_capital_weight',
 'Line_capital_weight',
 'Link_capital_weight',
 'Load_capital_weight',
 'Process_capital_weight',
 'StorageUnit_capital_weight',
 'Store_capital_weight',
 'Transformer_capital_weight']

The list has `Load_capital_weight`, although Load does not declare `capacity`.
`_providers` only checks the placeholders, not the trait list.
This is harmless: the spec never asks for that key, so it is never computed.

## 2.5 How a key is resolved

`NetworkSources.derivation(name)` tries two things, in order:

1. A registered provider for `name`.
2. A PyPSA attribute: a parameter named `<Component>_<attr>` is read from `c.da`.

If neither applies, it raises `NotImplementedError("no data prep for …")`.

In [32]:
src = sources(toy())
for key in ("Generator_capital_weight", "Generator_marginal_cost", "period_weighting_objective"):
    derive = src.derivation(key)
    print(f"{key:28} -> {getattr(derive.func, '__qualname__', derive.func)}")

Generator_capital_weight     -> _capital_weight
Generator_marginal_cost      -> NetworkSources.attribute
period_weighting_objective   -> _period_weighting


## 2.6 Caches

Rendering the YAML files takes time, so `build.py` caches on three levels:

| Cache | Key | Size |
|---|---|---|
| `_rendered`, `_base` | `linearized_unit_commitment` | one per value |
| `_PATCHED` | content of piecewise options and overrides (as JSON) | at most `KEEP = 8` |
| `NetworkSources.memo` | source name | one build |

An override in a dict and the same override as YAML text give the same cached spec:

In [33]:
print(spec() is spec())
print(spec(overrides=[HALF]) is spec(overrides=[yaml.safe_dump(HALF)]))
print(spec(overrides=[HALF]) is spec())

True
True
False


## 2.7 After linopy: make the model look like PyPSA's

`Model.from_spec` returns a model with spec names and spec dimensions.
`build_model` then runs three steps so that existing user code keeps working:

| Step | What it does |
|---|---|
| `_relabel` | Renames dims as PyPSA does: the unit dim (`Generator`) becomes `name`. Puts back the network's snapshot labels. Drops entries without rows. |
| `_scale` | Applies `Bus.scaling`, `GlobalConstraint.scaling` and `objective_scaling` to columns and rows. It does nothing when all factors are 1. |
| `_split_by_label` | Splits each global constraint into one `GlobalConstraint-<label>`, as before. |

The renaming of dims:

In [34]:
n = toy()
m = n.optimize.create_model()
print("spec dims: ", spec().variables["Generator_p"].dims)
print("model dims:", m.variables["Generator-p"].dims)

spec dims:  ('Generator', 'scenario', 'snapshot')
model dims: ('name', 'scenario', 'snapshot')


The split of global constraints. The spec states them by sense (`GlobalConstraint_upper`, ...),
but the model shows one constraint per label:

In [35]:
n = toy()
n.add("GlobalConstraint", "co2_limit", carrier_attribute="co2_emissions", sense="<=", constant=55)
n.optimize(log_to_console=False)
print([c for c in n.model.constraints if c.startswith("GlobalConstraint")])
n.global_constraints[["constant", "mu"]]

['GlobalConstraint-co2_limit']


,constant,mu
name,,
co2_limit,55.0,-806.25


A constraint can name its scaling rule in the `scaling:` field of its YAML entry.
A constraint without one gets a rule by its kind (see `_rule` in `build.py`). The explicit ones:

In [36]:
from pypsa.optimization.spec.build import spec_scalings

pd.Series(spec_scalings()).value_counts().rename("constraints per rule")

none         57
global        3
objective     2
cycle         1
carrier       1
Name: constraints per rule, dtype: int64

## 2.8 Checks before the build

`checks.check()` runs before the spec is built. It raises the same warnings and errors
as the old code did. So the YAML holds math only, and no input validation.

In [37]:
from pypsa.optimization.spec import checks

print(inspect.getsource(checks.check))

def check(
    n: Network,
    sns: pd.Index,
    *,
    multi_invest: bool = False,
    linearized_unit_commitment: bool = False,
    piecewise_options: Sequence[PiecewiseOptions] = (),
    **_: Any,
) -> None:
    """Warn and raise on network data as PyPSA's code-built model did, before the build."""
    _check_invariants(n)
    _check_deprecations(n)
    _check_modules(n)
    if linearized_unit_commitment:
        _check_tightening(n)
    _check_cyclic(n, multi_invest)
    _check_energy_limits(n)
    _check_piecewise(n, piecewise_options)
    _check_objective(n, sns)



## 2.9 Docs from the spec

`_docs.pages()` renders the spec as Markdown pages, one per trait and per component.
The formulation pages in the docs are generated, so they cannot drift from the code.

In [38]:
from pypsa.optimization.spec import _docs

pages = _docs.pages()
print(len(pages), "pages")
print("\n".join(pages))

22 pages
index.md
components/generator.md
components/load.md
components/link.md
components/store.md
components/storage_unit.md
components/line.md
components/process.md
components/transformer.md
traits/capacity.md
traits/commitment.md
traits/dispatch.md
traits/maintenance.md
traits/marginal_cost.md
traits/multiport.md
traits/oneport.md
traits/operational_limit.md
traits/passive_branch.md
traits/ramping.md
traits/rating.md
traits/storage.md
traits/transmission_expansion.md


In [39]:
print(pages["traits/capacity.md"][:1200])

# capacity

The trait `capacity` states one formulation, in `traits/capacity.yaml`, for each component that declares it. This page shows it for [`Generator`](../components/generator.md). Each component gives the trait these names:

| Component | Unit dimension | Nominal capacity | Power variable | Storage level | Cyclic flag | Port coefficient |
| --- | --- | --- | --- | --- | --- | --- |
| `Generator` | `Generator` | `p_nom` | `p` |  |  |  |
| `Link` | `Link` | `p_nom` | `p` |  |  | `efficiency` |
| `Store` | `Store` | `e_nom` | `p` | `e` | `e_cyclic` |  |
| `StorageUnit` | `StorageUnit` | `p_nom` | `p_dispatch` | `state_of_charge` | `cyclic_state_of_charge` |  |
| `Line` | `Line` | `s_nom` |  |  |  |  |
| `Process` | `Process` | `p_nom` | `p` |  |  | `rate` |
| `Transformer` | `Transformer` | `s_nom` |  |  |  |  |

## Variables

### `Generator_p_nom_ext` {#Generator_p_nom_ext}

*Variable.* the nominal capacity of an extendable asset

=== "Math"

    $$
    G_{i} \in \mathbb{R} \qquad

## Summary

- **To learn what the model is**, read the YAML files.
- **To learn where a number comes from**, search for its key in a `@source(...)` call.
- **To learn why the model looks like PyPSA's old one**, read the post-build steps in `build.py`.